# GOV-01 V2 Multi-label E6: strengthened N-RDD2024 MobileNetV2 baseline for Colab or Kaggle

This is a new experiment using the separate 39,617-image expansion split. It uses ImageNet-pretrained MobileNetV2 as a starting visual feature extractor, then learns a new seven-condition road classifier. It does not continue the old E2 model.

Only `train` and `validation` are read. The new `test` folder is deliberately never loaded in this notebook.

In [ ]:
# Data setup. In Colab, upload the ZIP to Google Drive. In Kaggle, attach it as a private Dataset.
from pathlib import Path
import csv, json, random, shutil, time, zipfile

DATASET_FOLDER = 'v2_multilabel_n_rdd_expansion'
CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover', 'unpaved_road', 'road_marking', 'speed_bump')
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

def has_training_layout(root):
    return all((root / split / 'images').is_dir() and (root / split / 'labels.csv').is_file() for split in ('train', 'validation'))

try:
    from google.colab import drive
    RUNTIME = 'colab'
except ImportError:
    RUNTIME = 'kaggle'

if RUNTIME == 'colab':
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/GOV-01')
    WORK_ROOT = Path('/content')
    archive_path = DRIVE_ROOT / f'{DATASET_FOLDER}.zip'
    if not archive_path.is_file():
        raise FileNotFoundError(f'Upload {DATASET_FOLDER}.zip to this Google Drive folder: {DRIVE_ROOT}')
else:
    INPUT_ROOT = Path('/kaggle/input')
    WORK_ROOT = Path('/kaggle/working')
    attached_folders = [path for path in INPUT_ROOT.rglob(DATASET_FOLDER) if path.is_dir() and has_training_layout(path)]
    attached_archives = list(INPUT_ROOT.rglob(f'{DATASET_FOLDER}.zip'))
    if len(attached_folders) == 1:
        DATA_ROOT = attached_folders[0]
        archive_path = None
        print('Using attached extracted Kaggle dataset:', DATA_ROOT)
    elif len(attached_archives) == 1:
        archive_path = attached_archives[0]
    else:
        raise FileNotFoundError(f'Attach exactly one Kaggle Dataset containing {DATASET_FOLDER}.zip or the extracted {DATASET_FOLDER} folder.')

LOCAL_DATA_ROOT = WORK_ROOT / DATASET_FOLDER
OUTPUT_DIRECTORY = WORK_ROOT / 'v2_multilabel_e6_n_rdd_mobilenetv2_output'
if archive_path is not None:
    if LOCAL_DATA_ROOT.exists() and not has_training_layout(LOCAL_DATA_ROOT):
        shutil.rmtree(LOCAL_DATA_ROOT)
    if not has_training_layout(LOCAL_DATA_ROOT):
        print('Extracting dataset ZIP to temporary runtime storage. This can take several minutes.')
        with zipfile.ZipFile(archive_path) as archive:
            destination_root = WORK_ROOT.resolve()
            for member in archive.infolist():
                destination = (WORK_ROOT / member.filename).resolve()
                if destination != destination_root and destination_root not in destination.parents:
                    raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
            archive.extractall(WORK_ROOT)
    DATA_ROOT = LOCAL_DATA_ROOT
    print('Using extracted dataset:', DATA_ROOT)

if not has_training_layout(DATA_ROOT):
    raise RuntimeError('Training/validation layout is incomplete. Reattach the complete expansion dataset and run from the first cell.')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print('Output folder:', OUTPUT_DIRECTORY)
print('Protected test is deliberately not loaded.')


In [ ]:
# Imports and GPU check. Enable a GPU in Colab or Kaggle, then run from the first cell.
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score, precision_recall_fscore_support

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus)
if not gpus:
    raise RuntimeError('No GPU is connected. In Colab choose Runtime -> Change runtime type -> T4 GPU; in Kaggle choose Settings -> Accelerator -> GPU. Then restart and run from the first cell.')


In [ ]:
# Read labels. Mask 0 means the source cannot confirm yes or no, so that label is ignored.
def read_split(split_name):
    split_root = DATA_ROOT / split_name
    paths, labels, masks = [], [], []
    with (split_root / 'labels.csv').open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            relative_image = Path(row['materialized_image'].replace(chr(92), '/'))
            image_path = split_root / relative_image
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing materialized image: {image_path}')
            label_row, mask_row = [], []
            for condition in CONDITIONS:
                known = int(row[f'{condition}_known'])
                value = row[f'{condition}_present']
                if known not in (0, 1) or (known == 0 and value != '') or (known == 1 and value not in ('0', '1')):
                    raise ValueError(f'Invalid masked label for {condition}: {row["record_id"]}')
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path))
            labels.append(label_row)
            masks.append(mask_row)
    return np.asarray(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32')

train_paths, train_labels, train_masks = read_split('train')
validation_paths, validation_labels, validation_masks = read_split('validation')
for name, labels, masks in [('Train', train_labels, train_masks), ('Validation', validation_labels, validation_masks)]:
    print(f'\n{name}: {len(labels)} images')
    for index, condition in enumerate(CONDITIONS):
        print(f'{condition:16} known={int(masks[:, index].sum()):5} positive={int((labels[:, index] * masks[:, index]).sum()):5}')


In [ ]:
# Build input pipelines. decode_image supports both JPG and PNG source images.
def load_image(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32)

def make_dataset(paths, labels, masks, training):
    packed = np.concatenate([labels, masks], axis=1)
    dataset = tf.data.Dataset.from_tensor_slices((paths, packed))
    if training:
        dataset = dataset.shuffle(min(len(paths), 8192), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(lambda path, target: (load_image(path), target), num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train_paths, train_labels, train_masks, training=True)
validation_dataset = make_dataset(validation_paths, validation_labels, validation_masks, training=False)
validation_images = validation_dataset.map(lambda images, packed: images)


In [ ]:
# Unknown labels receive neither loss nor score.
@tf.keras.utils.register_keras_serializable(package='gov01')
def masked_binary_crossentropy(packed_targets, probabilities):
    count = len(CONDITIONS)
    labels, masks = packed_targets[:, :count], packed_targets[:, count:]
    losses = tf.keras.backend.binary_crossentropy(labels, probabilities)
    return tf.math.divide_no_nan(tf.reduce_sum(losses * masks, axis=1), tf.reduce_sum(masks, axis=1))

class ValidationMaskedMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, images, labels, masks):
        super().__init__()
        self.images, self.labels, self.masks, self.history = images, labels, masks, []
    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.images, verbose=0)
        scores = [f1_score(self.labels[self.masks[:, i].astype(bool), i], probabilities[self.masks[:, i].astype(bool), i] >= 0.5, zero_division=0) for i in range(len(CONDITIONS))]
        score = float(np.mean(scores))
        self.history.append(score)
        print(f' validation masked macro F1 at threshold 0.50: {score:.4f}')


In [ ]:
# Stage 1: train a new road-condition head while MobileNetV2 stays frozen.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', seed=SEED),
    tf.keras.layers.RandomRotation(0.05, seed=SEED),
    tf.keras.layers.RandomContrast(0.10, seed=SEED),
], name='training_only_augmentation')
base = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3))
base.trainable = False
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='road_image')
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30, seed=SEED)(x)
outputs = tf.keras.layers.Dense(len(CONDITIONS), activation='sigmoid', name='condition_probabilities')(x)
model = tf.keras.Model(inputs, outputs, name='v2_multilabel_e6_n_rdd_mobilenetv2')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=masked_binary_crossentropy)
head_callback = ValidationMaskedMacroF1(validation_images, validation_labels, validation_masks)
head_checkpoint = OUTPUT_DIRECTORY / 'e6_head_best.keras'
head_history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5, callbacks=[head_callback, tf.keras.callbacks.ModelCheckpoint(head_checkpoint, monitor='val_loss', save_best_only=True)], verbose=1)


In [ ]:
# Stage 2: gently fine-tune only the last 30 MobileNetV2 layers. Batch-normalization layers stay frozen.
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss=masked_binary_crossentropy)
fine_callback = ValidationMaskedMacroF1(validation_images, validation_labels, validation_masks)
fine_checkpoint = OUTPUT_DIRECTORY / 'e6_n_rdd_mobilenetv2_best.keras'
fine_history = model.fit(train_dataset, validation_data=validation_dataset, epochs=15, callbacks=[fine_callback, tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True), tf.keras.callbacks.ModelCheckpoint(fine_checkpoint, monitor='val_loss', save_best_only=True)], verbose=1)


In [ ]:
# Save validation evidence only. Do not load the new test folder here.
best_model = tf.keras.models.load_model(fine_checkpoint, compile=False)
probabilities = best_model.predict(validation_images, verbose=1)
per_condition = {}
for index, condition in enumerate(CONDITIONS):
    known = validation_masks[:, index].astype(bool)
    precision, recall, f1, _ = precision_recall_fscore_support(validation_labels[known, index], probabilities[known, index] >= 0.5, average='binary', zero_division=0)
    per_condition[condition] = {'known_images': int(known.sum()), 'positive_images': int(validation_labels[known, index].sum()), 'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}
metrics = {'experiment': 'v2_multilabel_e6_n_rdd_mobilenetv2', 'selection_data': 'validation only', 'protected_test_loaded': False, 'model': 'ImageNet-pretrained MobileNetV2 with a new seven-label head', 'threshold': 0.5, 'masked_validation_macro_f1': float(np.mean([item['f1'] for item in per_condition.values()])), 'per_condition': per_condition, 'head_epochs_completed': len(head_history.history['loss']), 'fine_tune_epochs_completed': len(fine_history.history['loss'])}
(OUTPUT_DIRECTORY / 'e6_validation_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
best_model.save(OUTPUT_DIRECTORY / 'e6_n_rdd_mobilenetv2_final.keras')
print(json.dumps(metrics, indent=2))
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(head_history.history['loss'] + fine_history.history['loss'], label='train loss')
plt.plot(head_history.history['val_loss'] + fine_history.history['val_loss'], label='validation loss')
plt.axvline(len(head_history.history['loss']) - 0.5, color='gray', linestyle='--', label='fine-tuning starts')
plt.xlabel('epoch'); plt.ylabel('masked loss'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(head_callback.history + fine_callback.history, label='validation masked macro F1')
plt.xlabel('epoch'); plt.ylabel('F1'); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / 'e6_learning_curve.png', dpi=150)
plt.show()
result_zip = WORK_ROOT / 'v2_multilabel_e6_n_rdd_mobilenetv2_output.zip'
if result_zip.exists(): result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_DIRECTORY)
if RUNTIME == 'colab':
    drive_output = DRIVE_ROOT / result_zip.name
    shutil.copy2(result_zip, drive_output)
    print('Validation-only training output copied to Google Drive:', drive_output)
else:
    print('Validation-only training output:', result_zip)
print('The new protected test is still untouched.')
